In [10]:
%%writefile /kaggle/working/setup.py
# Cell 1: Setup and Imports
# Run this first to install dependencies and import required libraries

# Install required packages
# !pip install diffusers==0.21.4
# !pip install transformers==4.35.0
# !pip install accelerate==0.24.1
# !pip install xformers==0.0.22
# !pip install opencv-python==4.8.1.78
# !pip install pillow==10.0.1
# !pip install datasets==2.14.6
# !pip install peft==0.6.2
# !pip install bitsandbytes==0.41.2.post2

import os
import torch
import numpy as np
from PIL import Image, ImageDraw, ImageFont, ImageOps
import cv2
import json
from pathlib import Path
from typing import List, Dict, Tuple, Optional
import pandas as pd
from tqdm import tqdm
import random
import re
import shutil
from datetime import datetime

# Deep learning imports
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from diffusers import (
    StableDiffusionPipeline, 
    UNet2DConditionModel, 
    DDPMScheduler,
    AutoencoderKL
)
from transformers import CLIPTextModel, CLIPTokenizer
from accelerate import Accelerator
from peft import LoraConfig, get_peft_model, TaskType
import logging

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Check GPU availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

Overwriting /kaggle/working/setup.py


In [11]:
%%writefile /kaggle/working/config.py
#!/usr/bin/env python3
# config.py - Configuration settings for calligraphy LoRA training
# This file contains all configuration settings for both simple and complex calligraphy styles

import torch
from pathlib import Path
from typing import List, Dict, Any, Optional
import os

class BaseConfig:
    """Base configuration class with common settings"""
    
    def __init__(self):
        # Hardware settings
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.mixed_precision = "fp16" if torch.cuda.is_available() else "no"
        self.use_8bit_adam = True if torch.cuda.is_available() else False
        
        # Model settings
        self.model_name = "runwayml/stable-diffusion-v1-5"
        self.revision = "fp16" if torch.cuda.is_available() else "main"
        self.torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
        
        # Image settings
        self.resolution = 512  # Standard SD resolution
        self.center_crop = True
        self.random_flip = False  # Usually False for text/calligraphy
        
        # Dataset settings
        self.validation_split = 0.1
        self.max_train_samples = None  # None means use all samples
        self.preprocessing_num_workers = 4 if torch.cuda.is_available() else 2
        
        # Output settings
        self.output_base_dir = "/kaggle/working"
        self.save_precision = "fp16" if torch.cuda.is_available() else "fp32"
        self.save_model_as = "safetensors"  # or "ckpt"
        
        # Logging settings
        self.logging_level = "INFO"
        self.log_with = "tensorboard"  # "wandb", "tensorboard", or None
        self.report_to = None  # "wandb" if you want to use wandb
        
        # Safety settings
        self.enable_xformers_memory_efficient_attention = True
        self.gradient_checkpointing = True  # Saves memory at cost of speed
        
    def to_dict(self) -> Dict[str, Any]:
        """Convert configuration to dictionary"""
        return {k: v for k, v in self.__dict__.items() if not k.startswith('_')}

class SimpleCalligraphyConfig(BaseConfig):
    """Configuration for simple calligraphy styles (elegant, cursive, etc.)"""
    
    def __init__(self, style_name: str = "elegant_calligraphy"):
        super().__init__()
        
        # Style settings
        self.style_name = style_name
        self.style_type = "simple"
        
        # LoRA settings - Conservative for simple styles
        self.lora_rank = 4
        self.lora_alpha = 32
        self.lora_dropout = 0.1
        self.lora_target_modules = [
            "to_k", "to_q", "to_v", "to_out.0",
            "ff.net.0.proj", "ff.net.2"
        ]
        
        # Training settings - Optimized for simple styles
        self.batch_size = 1
        self.gradient_accumulation_steps = 4  # Effective batch size = 4
        self.learning_rate = 1e-4
        self.lr_scheduler = "cosine"
        self.lr_warmup_steps = 500
        self.max_epochs = 15
        self.max_train_steps = None  # Will be calculated
        
        # Optimization settings
        self.optimizer_type = "AdamW8bit" if self.use_8bit_adam else "AdamW"
        self.adam_beta1 = 0.9
        self.adam_beta2 = 0.999
        self.adam_weight_decay = 1e-2
        self.adam_epsilon = 1e-8
        self.max_grad_norm = 1.0
        
        # Noise and generation settings - Gentle for simple styles
        self.noise_offset = 0.1
        self.input_perturbation = 0.0
        self.snr_gamma = 5.0
        self.prediction_type = "epsilon"
        
        # Data augmentation - Minimal for text
        self.random_crop = False
        self.color_jitter = False
        
        # Validation settings
        self.validation_epochs = 5
        self.validation_steps = 250
        self.num_validation_images = 4
        self.validation_prompts = [
            f"a {style_name} calligraphy letter 'a', black ink on white background, elegant handwriting",
            f"a {style_name} calligraphy word 'hello', black ink on white background, flowing script",
            f"a {style_name} calligraphy letter 'g', black ink on white background, cursive style",
            f"a {style_name} calligraphy word 'world', black ink on white background, beautiful lettering"
        ]
        
        # Checkpointing
        self.save_steps = 500
        self.save_every_n_epochs = 10
        self.keep_only_last_checkpoint = False
        
        # Memory optimization for Kaggle
        self.enable_cpu_offload = False  # Can enable if running out of memory
        self.use_cpu_text_encoder = False
        self.pre_compute_text_embeddings = True

class ComplexCalligraphyConfig(BaseConfig):
    """Configuration for complex calligraphy styles (gothic, ornate, etc.)"""
    
    def __init__(self, style_name: str = "gothic_calligraphy"):
        super().__init__()
        
        # Style settings
        self.style_name = style_name
        self.style_type = "complex"
        
        # LoRA settings - More aggressive for complex styles
        self.lora_rank = 8  # Higher rank for complex patterns
        self.lora_alpha = 64  # Higher alpha for stronger adaptation
        self.lora_dropout = 0.15  # Slightly higher dropout
        self.lora_target_modules = [
            "to_k", "to_q", "to_v", "to_out.0",
            "ff.net.0.proj", "ff.net.2",
            "conv_in", "conv_out"  # Additional modules for complex styles
        ]
        
        # Training settings - More intensive for complex styles
        self.batch_size = 1
        self.gradient_accumulation_steps = 8  # Effective batch size = 8
        self.learning_rate = 5e-5  # Lower learning rate for stability
        self.lr_scheduler = "cosine_with_restarts"
        self.lr_warmup_steps = 1000  # Longer warmup
        self.max_epochs = 100  # More epochs for complex styles
        self.max_train_steps = None
        
        # Optimization settings
        self.optimizer_type = "AdamW8bit" if self.use_8bit_adam else "AdamW"
        self.adam_beta1 = 0.9
        self.adam_beta2 = 0.999
        self.adam_weight_decay = 1e-2
        self.adam_epsilon = 1e-8
        self.max_grad_norm = 1.0
        
        # Noise settings - More aggressive for complex styles
        self.noise_offset = 0.15  # Higher noise offset
        self.input_perturbation = 0.1  # Add input perturbation
        self.snr_gamma = 3.0  # Lower SNR gamma for more aggressive training
        self.prediction_type = "epsilon"
        
        # Data augmentation - Still minimal for text
        self.random_crop = False
        self.color_jitter = False
        
        # Validation settings
        self.validation_epochs = 10
        self.validation_steps = 500
        self.num_validation_images = 6
        self.validation_prompts = [
            f"a {style_name} gothic calligraphy letter 'd', black ink on white background, ornate medieval style",
            f"a {style_name} gothic calligraphy word 'dog', black ink on white background, decorative lettering",
            f"a {style_name} gothic calligraphy letter 'A', black ink on white background, elaborate flourishes",
            f"a {style_name} gothic calligraphy word 'fate', black ink on white background, medieval manuscript style",
            f"a {style_name} gothic calligraphy letter 'f', black ink on white background, ornamental design",
            f"a {style_name} gothic calligraphy word 'dreams', black ink on white background, gothic script"
        ]
        
        # Checkpointing - More frequent for complex training
        self.save_steps = 250
        self.save_every_n_epochs = 5
        self.keep_only_last_checkpoint = False
        
        # Memory optimization
        self.enable_cpu_offload = True  # More likely needed for complex training
        self.use_cpu_text_encoder = False
        self.pre_compute_text_embeddings = True
        self.gradient_checkpointing = True

class KaggleOptimizedConfig:
    """Additional optimizations specifically for Kaggle environment"""
    
    @staticmethod
    def apply_kaggle_optimizations(config: BaseConfig) -> BaseConfig:
        """Apply Kaggle-specific optimizations"""
        
        # Memory optimizations
        config.gradient_checkpointing = True
        config.enable_xformers_memory_efficient_attention = True
        config.pre_compute_text_embeddings = True
        
        # Batch size optimizations for limited memory
        if config.style_type == "complex":
            config.batch_size = 1
            config.gradient_accumulation_steps = 8
        else:
            config.batch_size = 1
            config.gradient_accumulation_steps = 4
        
        # Use 8-bit optimizers to save memory
        config.use_8bit_adam = True
        config.optimizer_type = "AdamW8bit"
        
        # Enable CPU offloading if needed
        if hasattr(config, 'enable_cpu_offload'):
            config.enable_cpu_offload = True
        
        # Adjust resolution if needed (can lower to 448 or 384 to save memory)
        # config.resolution = 448  # Uncomment if running out of memory
        
        return config

class DataConfig:
    """Configuration for data preparation and processing"""
    
    def __init__(self):
        # Data paths (will be set by main script)
        self.character_images_dir = None
        self.word_images_dir = None
        self.output_data_dir = None
        
        # Image processing settings
        self.target_size = (512, 512)
        self.background_color = (255, 255, 255)  # White background
        self.text_color = (0, 0, 0)  # Black text
        self.padding = 50  # Padding around text
        
        # Data augmentation settings
        self.enable_augmentation = True
        self.rotation_range = 5  # degrees
        self.brightness_range = 0.1
        self.contrast_range = 0.1
        self.noise_factor = 0.02
        
        # Text processing
        self.min_text_length = 1
        self.max_text_length = 50
        self.filter_non_alphabetic = False
        
        # Dataset splitting
        self.train_split = 0.8
        self.val_split = 0.1
        self.test_split = 0.1
        
        # Caching
        self.cache_latents = True  # Cache VAE latents to speed up training
        self.cache_text_embeddings = True

def get_config(style_type: str = "simple", 
               style_name: str = None,
               apply_kaggle_optimizations: bool = True) -> BaseConfig:
    """
    Factory function to get appropriate configuration
    
    Args:
        style_type: "simple" or "complex"
        style_name: Name of the calligraphy style
        apply_kaggle_optimizations: Whether to apply Kaggle-specific optimizations
    
    Returns:
        Configured config object
    """
    
    if style_type.lower() == "simple":
        if style_name is None:
            style_name = "elegant_calligraphy"
        config = SimpleCalligraphyConfig(style_name)
    elif style_type.lower() == "complex":
        if style_name is None:
            style_name = "gothic_calligraphy"
        config = ComplexCalligraphyConfig(style_name)
    else:
        raise ValueError(f"Unknown style_type: {style_type}. Must be 'simple' or 'complex'")
    
    # Apply Kaggle optimizations if requested
    if apply_kaggle_optimizations:
        config = KaggleOptimizedConfig.apply_kaggle_optimizations(config)
    
    return config

def create_experiment_config(character_dir: str,
                           word_dir: str,
                           style_name: str,
                           style_type: str = "simple",
                           output_dir: str = "/kaggle/working") -> Dict[str, Any]:
    """
    Create complete experiment configuration
    
    Args:
        character_dir: Path to character images directory
        word_dir: Path to word images directory
        style_name: Name of the calligraphy style
        style_type: "simple" or "complex"
        output_dir: Base output directory
    
    Returns:
        Complete experiment configuration dictionary
    """
    
    # Get main config
    main_config = get_config(style_type, style_name, apply_kaggle_optimizations=True)
    
    # Create data config
    data_config = DataConfig()
    data_config.character_images_dir = character_dir
    data_config.word_images_dir = word_dir
    data_config.output_data_dir = os.path.join(output_dir, f"{style_name}_processed_data")
    
    # Update main config with paths
    main_config.output_base_dir = output_dir
    main_config.character_dir = character_dir
    main_config.word_dir = word_dir
    
    return {
        "main_config": main_config,
        "data_config": data_config,
        "experiment_info": {
            "style_name": style_name,
            "style_type": style_type,
            "character_dir": character_dir,
            "word_dir": word_dir,
            "output_dir": output_dir
        }
    }

# Predefined style configurations
STYLE_PRESETS = {
    "elegant_script": {
        "type": "simple",
        "description": "Elegant flowing script calligraphy",
        "recommended_epochs": 50,
        "difficulty": "easy"
    },
    "modern_calligraphy": {
        "type": "simple", 
        "description": "Modern brush-style calligraphy",
        "recommended_epochs": 60,
        "difficulty": "easy"
    },
    "gothic_calligraphy": {
        "type": "complex",
        "description": "Medieval gothic style with ornate flourishes",
        "recommended_epochs": 100,
        "difficulty": "hard"
    },
    "blackletter": {
        "type": "complex",
        "description": "Traditional blackletter/textura style",
        "recommended_epochs": 120,
        "difficulty": "very_hard"
    },
    "copperplate": {
        "type": "simple",
        "description": "Classic copperplate business script",
        "recommended_epochs": 40,
        "difficulty": "medium"
    }
}

def get_style_preset(style_name: str) -> Dict[str, Any]:
    """Get predefined style configuration"""
    return STYLE_PRESETS.get(style_name, {
        "type": "simple",
        "description": "Custom calligraphy style",
        "recommended_epochs": 50,
        "difficulty": "medium"
    })

# Environment checks
def check_environment():
    """Check if environment is properly configured"""
    checks = {
        "torch_available": False,
        "cuda_available": False,
        "diffusers_available": False,
        "peft_available": False,
        "sufficient_memory": False
    }
    
    try:
        import torch
        checks["torch_available"] = True
        checks["cuda_available"] = torch.cuda.is_available()
        
        if torch.cuda.is_available():
            gpu_memory = torch.cuda.get_device_properties(0).total_memory / (1024**3)
            checks["sufficient_memory"] = gpu_memory >= 12  # At least 12GB recommended
        
        import diffusers
        checks["diffusers_available"] = True
        
        import peft
        checks["peft_available"] = True
        
    except ImportError as e:
        print(f"Import error: {e}")
    
    return checks

if __name__ == "__main__":
    # Example usage and testing
    print("=== Calligraphy LoRA Training Configuration ===")
    
    # Check environment
    env_checks = check_environment()
    print(f"Environment checks: {env_checks}")
    
    # Test simple configuration
    print("\n--- Simple Calligraphy Config ---")
    simple_config = get_config("simple", "elegant_script")
    print(f"Style: {simple_config.style_name}")
    print(f"LoRA Rank: {simple_config.lora_rank}")
    print(f"Learning Rate: {simple_config.learning_rate}")
    print(f"Max Epochs: {simple_config.max_epochs}")
    
    # Test complex configuration  
    print("\n--- Complex Calligraphy Config ---")
    complex_config = get_config("complex", "gothic_calligraphy")
    print(f"Style: {complex_config.style_name}")
    print(f"LoRA Rank: {complex_config.lora_rank}")
    print(f"Learning Rate: {complex_config.learning_rate}")
    print(f"Max Epochs: {complex_config.max_epochs}")
    
    # Test style presets
    print("\n--- Available Style Presets ---")
    for style, info in STYLE_PRESETS.items():
        print(f"{style}: {info['description']} ({info['type']}, {info['difficulty']})")
    
    print("\nConfiguration module loaded successfully!")

Overwriting /kaggle/working/config.py


In [12]:
%%writefile /kaggle/working/data_preparation.py
#!/usr/bin/env python3
# Cell 2: Data Preparation and Processing Functions

import cv2
import numpy as np
import re
import json
from pathlib import Path
import logging
from tqdm import tqdm
from PIL import Image 
from typing import Dict

logger = logging.getLogger(__name__)


class ImageProcessor:
    """Handles image preprocessing for calligraphy training"""
    
    def __init__(self, target_size: int = 512):
        self.target_size = target_size
        
    def preprocess_image(self, image_path: str) -> Image.Image:
        """Preprocess calligraphy image for training"""
        img = Image.open(image_path).convert('RGB')
        
        # Convert to grayscale first for better text detection
        gray = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2GRAY)
        
        # Apply adaptive thresholding for better text extraction
        binary = cv2.adaptiveThreshold(
            gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 11, 2
        )
        
        # Convert back to PIL Image
        processed = Image.fromarray(binary).convert('RGB')
        
        # Resize while maintaining aspect ratio
        processed = self._resize_with_padding(processed)
        
        return processed
    
    def _resize_with_padding(self, img: Image.Image) -> Image.Image:
        """Resize image to target size with padding"""
        # Calculate aspect ratio
        w, h = img.size
        aspect = w / h
        
        if aspect > 1:  # Width > Height
            new_w = self.target_size
            new_h = int(self.target_size / aspect)
        else:  # Height >= Width
            new_h = self.target_size
            new_w = int(self.target_size * aspect)
        
        # Resize image
        img = img.resize((new_w, new_h), Image.Resampling.LANCZOS)
        
        # Create white background
        background = Image.new('RGB', (self.target_size, self.target_size), 'white')
        
        # Paste resized image in center
        x_offset = (self.target_size - new_w) // 2
        y_offset = (self.target_size - new_h) // 2
        background.paste(img, (x_offset, y_offset))
        
        return background

class DatasetBuilder:
    """Builds training dataset from character and word images"""
    
    def __init__(self, style_name: str = "calligraphy"):
        self.style_name = style_name
        self.processor = ImageProcessor()
        
    def extract_character_from_filename(self, filename: str) -> str:
        """Extract character from filename"""
        # Remove extension and clean filename
        char = Path(filename).stem
        # Handle special characters and clean up
        char = re.sub(r'[^a-zA-Z0-9]', '', char)
        return char.lower() if char else 'unknown'
    
    def extract_word_from_filename(self, filename: str) -> str:
        """Extract word from filename"""
        word = Path(filename).stem
        # Clean and normalize
        word = re.sub(r'[^a-zA-Z0-9\s]', '', word)
        return word.lower().strip() if word else 'unknown'
    
    def create_training_data(self, 
                           character_dir: str, 
                           word_dir: str, 
                           output_dir: str) -> Dict:
        """Create training dataset with proper structure"""
        
        output_path = Path(output_dir)
        images_dir = output_path / "images"
        metadata_path = output_path / "metadata.jsonl"
        
        # Create directories
        images_dir.mkdir(parents=True, exist_ok=True)
        
        training_data = []
        image_counter = 0
        
        # Process character images
        char_dir = Path(character_dir)
        if char_dir.exists():
            logger.info(f"Processing character images from {char_dir}")
            for img_file in tqdm(char_dir.glob("*.png"), desc="Processing characters"):
                try:
                    char = self.extract_character_from_filename(img_file.name)
                    if char and char != 'unknown':
                        # Process and save image
                        processed_img = self.processor.preprocess_image(str(img_file))
                        
                        # Save processed image
                        output_filename = f"char_{image_counter:06d}.png"
                        output_filepath = images_dir / output_filename
                        processed_img.save(output_filepath)
                        
                        # Create training prompt
                        prompt = f"a {self.style_name} style calligraphy letter '{char}', black ink on white background, high quality handwriting"
                        
                        training_data.append({
                            "file_name": output_filename,
                            "text": prompt,
                            "type": "character",
                            "content": char
                        })
                        
                        image_counter += 1
                        
                except Exception as e:
                    logger.warning(f"Error processing character {img_file}: {e}")
        
        # Process word images
        word_dir = Path(word_dir)
        if word_dir.exists():
            logger.info(f"Processing word images from {word_dir}")
            for img_file in tqdm(word_dir.glob("*.png"), desc="Processing words"):
                try:
                    word = self.extract_word_from_filename(img_file.name)
                    if word and word != 'unknown':
                        # Process and save image
                        processed_img = self.processor.preprocess_image(str(img_file))
                        
                        # Save processed image
                        output_filename = f"word_{image_counter:06d}.png"
                        output_filepath = images_dir / output_filename
                        processed_img.save(output_filepath)
                        
                        # Create training prompt
                        prompt = f"a {self.style_name} style calligraphy word '{word}', black ink on white background, elegant handwriting"
                        
                        training_data.append({
                            "file_name": output_filename,
                            "text": prompt,
                            "type": "word",
                            "content": word
                        })
                        
                        image_counter += 1
                        
                except Exception as e:
                    logger.warning(f"Error processing word {img_file}: {e}")
        
        # Save metadata
        with open(metadata_path, 'w') as f:
            for item in training_data:
                f.write(json.dumps(item) + '\n')
        
        logger.info(f"Created {len(training_data)} training samples")
        logger.info(f"Data saved to {output_path}")
        
        return {
            "total_samples": len(training_data),
            "images_dir": str(images_dir),
            "metadata_path": str(metadata_path),
            "training_data": training_data
        }

# Initialize the dataset builder
dataset_builder = DatasetBuilder(style_name="gothic_calligraphy")

print("Data preparation functions loaded successfully!")
print("Ready to process your character and word images.")

Overwriting /kaggle/working/data_preparation.py


In [13]:
%%writefile /kaggle/working/dataset.py
#!/usr/bin/env python3
# Cell 3: Custom Dataset Class for LoRA Training
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from pathlib import Path
import json
from PIL import Image
from transformers import CLIPTokenizer
import logging
from typing import Tuple

logger = logging.getLogger(__name__)


class CalligraphyDataset(Dataset):
    """Custom dataset for calligraphy LoRA training"""
    
    def __init__(self, 
                 images_dir: str,
                 metadata_path: str,
                 tokenizer: CLIPTokenizer,
                 resolution: int = 512,
                 flip_p: float = 0.5):
        
        self.images_dir = Path(images_dir)
        self.tokenizer = tokenizer
        self.resolution = resolution
        self.flip_p = flip_p
        
        # Load metadata
        self.data = []
        with open(metadata_path, 'r') as f:
            for line in f:
                self.data.append(json.loads(line.strip()))
        
        # Image transforms
        self.image_transforms = transforms.Compose([
            transforms.Resize((resolution, resolution), interpolation=transforms.InterpolationMode.BILINEAR),
            transforms.RandomHorizontalFlip(p=flip_p),
            transforms.ToTensor(),
            transforms.Normalize([0.5], [0.5])
        ])
        
        # Validation transforms (no augmentation)
        self.val_transforms = transforms.Compose([
            transforms.Resize((resolution, resolution), interpolation=transforms.InterpolationMode.BILINEAR),
            transforms.ToTensor(),
            transforms.Normalize([0.5], [0.5])
        ])
        
        logger.info(f"Loaded {len(self.data)} training samples")
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        
        # Load image
        image_path = self.images_dir / item["file_name"]
        image = Image.open(image_path).convert('RGB')
        
        # Apply transforms
        image = self.image_transforms(image)
        
        # Tokenize text
        text = item["text"]
        text_inputs = self.tokenizer(
            text,
            padding="max_length",
            max_length=self.tokenizer.model_max_length,
            truncation=True,
            return_tensors="pt"
        )
        
        return {
            "pixel_values": image,
            "input_ids": text_inputs.input_ids[0],
            "attention_mask": text_inputs.attention_mask[0],
            "text": text,
            "type": item["type"],
            "content": item["content"]
        }

class DataCollator:
    """Custom data collator for batch processing"""
    
    def __init__(self, tokenizer: CLIPTokenizer):
        self.tokenizer = tokenizer
    
    def __call__(self, examples):
        batch = {}
        batch["pixel_values"] = torch.stack([example["pixel_values"] for example in examples])
        batch["input_ids"] = torch.stack([example["input_ids"] for example in examples])
        batch["attention_mask"] = torch.stack([example["attention_mask"] for example in examples])
        
        return batch

def create_data_loaders(images_dir: str, 
                       metadata_path: str, 
                       tokenizer: CLIPTokenizer,
                       batch_size: int = 1,
                       validation_split: float = 0.1,
                       resolution: int = 512) -> Tuple[DataLoader, DataLoader]:
    """Create training and validation data loaders"""
    
    # Load all data
    full_dataset = CalligraphyDataset(
        images_dir=images_dir,
        metadata_path=metadata_path,
        tokenizer=tokenizer,
        resolution=resolution
    )
    
    # Split into train and validation
    dataset_size = len(full_dataset)
    val_size = int(dataset_size * validation_split)
    train_size = dataset_size - val_size
    
    train_dataset, val_dataset = torch.utils.data.random_split(
        full_dataset, [train_size, val_size]
    )
    
    # Create data collator
    collator = DataCollator(tokenizer)
    
    # Create data loaders
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=collator,
        num_workers=0,  # Set to 0 for Kaggle compatibility
        pin_memory=True if torch.cuda.is_available() else False
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collator,
        num_workers=0,
        pin_memory=True if torch.cuda.is_available() else False
    )
    
    logger.info(f"Created data loaders: {len(train_dataset)} train, {len(val_dataset)} validation samples")
    
    return train_loader, val_loader

def preview_dataset(dataset: CalligraphyDataset, num_samples: int = 3):
    """Preview dataset samples"""
    import matplotlib.pyplot as plt
    
    fig, axes = plt.subplots(1, num_samples, figsize=(15, 5))
    if num_samples == 1:
        axes = [axes]
    
    for i in range(min(num_samples, len(dataset))):
        sample = dataset[i]
        
        # Convert tensor back to image
        image = sample["pixel_values"]
        image = (image + 1.0) / 2.0  # Denormalize
        image = torch.clamp(image, 0, 1)
        image = transforms.ToPILImage()(image)
        
        axes[i].imshow(image)
        axes[i].set_title(f"Type: {sample['type']}\nContent: {sample['content']}")
        axes[i].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    # Print sample prompts
    print("\nSample prompts:")
    for i in range(min(3, len(dataset))):
        sample = dataset[i]
        print(f"{i+1}. {sample['text']}")

print("Custom dataset classes loaded successfully!")
print("Ready to create data loaders from your processed images.")

Overwriting /kaggle/working/dataset.py


In [14]:
%%writefile /kaggle/working/model_setup.py
#!/usr/bin/env python3
# Cell 4: LoRA Model Setup and Configuration
from typing import List
import torch
import logging
from transformers import CLIPTokenizer, CLIPTextModel
from diffusers import UNet2DConditionModel, AutoencoderKL, DDPMScheduler
from peft import get_peft_model, LoraConfig, TaskType
logger = logging.getLogger(__name__)


class LoRAModelSetup:
    """Setup and configure LoRA model for calligraphy training"""
    
    def __init__(self, 
                 model_name: str = "runwayml/stable-diffusion-v1-5",
                 revision: str = "fp16",
                 torch_dtype: torch.dtype = torch.float16):
        
        self.model_name = model_name
        self.revision = revision
        self.torch_dtype = torch_dtype
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        
    def load_models(self):
        """Load and setup all required models"""
        logger.info(f"Loading models from {self.model_name}")
        
        # Load tokenizer
        self.tokenizer = CLIPTokenizer.from_pretrained(
            self.model_name,
            subfolder="tokenizer",
            revision=self.revision,
            torch_dtype=self.torch_dtype
        )
        
        # Load text encoder
        self.text_encoder = CLIPTextModel.from_pretrained(
            self.model_name,
            subfolder="text_encoder",
            revision=self.revision,
            torch_dtype=self.torch_dtype
        )
        
        # Load VAE
        self.vae = AutoencoderKL.from_pretrained(
            self.model_name,
            subfolder="vae",
            revision=self.revision,
            torch_dtype=self.torch_dtype
        )
        
        # Load UNet
        self.unet = UNet2DConditionModel.from_pretrained(
            self.model_name,
            subfolder="unet",
            revision=self.revision,
            torch_dtype=self.torch_dtype
        )
        
        # Load scheduler
        self.scheduler = DDPMScheduler.from_pretrained(
            self.model_name,
            subfolder="scheduler"
        )
        
        # Move models to device
        self.text_encoder.to(self.device)
        self.vae.to(self.device)
        self.unet.to(self.device)
        
        # Set models to eval mode (except UNet which we'll train)
        self.text_encoder.eval()
        self.vae.eval()
        
        # Freeze parameters
        self.text_encoder.requires_grad_(False)
        self.vae.requires_grad_(False)
        self.unet.requires_grad_(False)
        
        logger.info("Models loaded successfully")
        
        return {
            "tokenizer": self.tokenizer,
            "text_encoder": self.text_encoder,
            "vae": self.vae,
            "unet": self.unet,
            "scheduler": self.scheduler
        }
    
    def setup_lora_config(self, 
                         rank: int = 4,
                         alpha: int = 32,
                         dropout: float = 0.1,
                         target_modules: List[str] = None):
        """Setup LoRA configuration"""
        
        if target_modules is None:
            # Target attention modules in UNet
            target_modules = [
                "to_k", "to_q", "to_v", "to_out.0",
                "ff.net.0.proj", "ff.net.2"
            ]
        
        lora_config = LoraConfig(
            r=rank,
            lora_alpha=alpha,
            target_modules=target_modules,
            lora_dropout=dropout,
            bias="none",
            task_type=TaskType.DIFFUSION
        )
        
        logger.info(f"LoRA config created with rank={rank}, alpha={alpha}")
        return lora_config
    
    def apply_lora_to_unet(self, lora_config: LoraConfig):
        """Apply LoRA to UNet model"""
        
        # Apply LoRA
        self.unet = get_peft_model(self.unet, lora_config)
        self.unet.print_trainable_parameters()
        
        logger.info("LoRA applied to UNet successfully")
        return self.unet

class TrainingConfig:
    """Training configuration and hyperparameters"""
    
    def __init__(self):
        # Model settings
        self.model_name = "runwayml/stable-diffusion-v1-5"
        self.revision = "fp16"
        self.resolution = 512
        
        # LoRA settings
        self.lora_rank = 4
        self.lora_alpha = 32
        self.lora_dropout = 0.1
        
        # Training settings
        self.batch_size = 1  # Small batch size for limited GPU memory
        self.gradient_accumulation_steps = 4  # Effective batch size = 4
        self.learning_rate = 1e-4
        self.max_epochs = 50
        self.save_steps = 500
        self.validation_steps = 250
        
        # Optimization settings
        self.adam_beta1 = 0.9
        self.adam_beta2 = 0.999
        self.adam_weight_decay = 1e-2
        self.adam_epsilon = 1e-8
        self.max_grad_norm = 1.0
        
        # Noise settings
        self.noise_offset = 0.1
        self.snr_gamma = 5.0
        
        # Output settings
        self.output_dir = "/kaggle/working/calligraphy_lora"
        self.logging_dir = "/kaggle/working/logs"
        self.mixed_precision = "fp16"
        
        # Validation settings
        self.validation_split = 0.1
        self.num_validation_images = 4
        
    def to_dict(self):
        """Convert config to dictionary"""
        return {k: v for k, v in self.__dict__.items() if not k.startswith('_')}

def calculate_memory_usage():
    """Calculate and display memory usage"""
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3
        reserved = torch.cuda.memory_reserved() / 1024**3
        total = torch.cuda.get_device_properties(0).total_memory / 1024**3
        
        print(f"GPU Memory Usage:")
        print(f"  Allocated: {allocated:.2f} GB")
        print(f"  Reserved: {reserved:.2f} GB")
        print(f"  Total: {total:.2f} GB")
        print(f"  Free: {total - reserved:.2f} GB")
        
        return {
            "allocated": allocated,
            "reserved": reserved,
            "total": total,
            "free": total - reserved
        }
    else:
        print("CUDA not available")
        return None

# Initialize configuration
config = TrainingConfig()
model_setup = LoRAModelSetup(
    model_name=config.model_name,
    revision=config.revision,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
)

print("LoRA model setup classes loaded successfully!")
print(f"Configuration: {config.to_dict()}")
print("\nMemory status:")
calculate_memory_usage()

Overwriting /kaggle/working/model_setup.py


In [15]:
%%writefile /kaggle/working/trainer.py
#!/usr/bin/env python3
# Cell 5: Training Loop and Optimization


import torch.nn.functional as F
from accelerate import Accelerator
from diffusers.optimization import get_scheduler
import wandb
from datetime import datetime
import math
import matplotlib.pyplot as plt
from model_setup import TrainingConfig

class CalligraphyTrainer:
    """Main trainer class for calligraphy LoRA training"""
    
    def __init__(self, config: TrainingConfig, models: dict):
        self.config = config
        self.models = models
        
        # Initialize accelerator
        self.accelerator = Accelerator(
            gradient_accumulation_steps=config.gradient_accumulation_steps,
            mixed_precision=config.mixed_precision,
            log_with="tensorboard",
            project_dir=config.logging_dir
        )
        
        # Extract models
        self.tokenizer = models["tokenizer"]
        self.text_encoder = models["text_encoder"]
        self.vae = models["vae"]
        self.unet = models["unet"]
        self.scheduler = models["scheduler"]
        
        # Setup weight dtype
        self.weight_dtype = torch.float32
        if self.accelerator.mixed_precision == "fp16":
            self.weight_dtype = torch.float16
        elif self.accelerator.mixed_precision == "bf16":
            self.weight_dtype = torch.bfloat16
        
        # Move models to correct dtype
        self.text_encoder.to(self.accelerator.device, dtype=self.weight_dtype)
        self.vae.to(self.accelerator.device, dtype=self.weight_dtype)
        
        # Training metrics
        self.training_losses = []
        self.validation_losses = []
        self.global_step = 0
        
        logger.info("Trainer initialized successfully")
    
    def setup_optimizer_and_scheduler(self, train_dataloader):
        """Setup optimizer and learning rate scheduler"""
        
        # Setup optimizer
        optimizer_class = torch.optim.AdamW
        optimizer = optimizer_class(
            self.unet.parameters(),
            lr=self.config.learning_rate,
            betas=(self.config.adam_beta1, self.config.adam_beta2),
            weight_decay=self.config.adam_weight_decay,
            eps=self.config.adam_epsilon
        )
        
        # Calculate total training steps
        num_update_steps_per_epoch = math.ceil(
            len(train_dataloader) / self.config.gradient_accumulation_steps
        )
        max_train_steps = self.config.max_epochs * num_update_steps_per_epoch
        
        # Setup scheduler
        lr_scheduler = get_scheduler(
            "cosine",
            optimizer=optimizer,
            num_warmup_steps=500,
            num_training_steps=max_train_steps
        )
        
        # Prepare with accelerator
        self.unet, optimizer, train_dataloader, lr_scheduler = self.accelerator.prepare(
            self.unet, optimizer, train_dataloader, lr_scheduler
        )
        
        return optimizer, lr_scheduler, max_train_steps
    
    def encode_prompt(self, prompt):
        """Encode text prompt to embeddings"""
        text_inputs = self.tokenizer(
            prompt,
            padding="max_length",
            max_length=self.tokenizer.model_max_length,
            truncation=True,
            return_tensors="pt"
        )
        
        with torch.no_grad():
            text_embeddings = self.text_encoder(
                text_inputs.input_ids.to(self.accelerator.device)
            )[0]
        
        return text_embeddings
    
    def compute_loss(self, batch):
        """Compute training loss"""
        # Get the text embedding for conditioning
        encoder_hidden_states = self.encode_prompt(batch["text"])
        
        # Convert images to latent space
        latents = self.vae.encode(batch["pixel_values"].to(dtype=self.weight_dtype)).latent_dist.sample()
        latents = latents * self.vae.config.scaling_factor
        
        # Sample noise to add to the latents
        noise = torch.randn_like(latents)
        if self.config.noise_offset:
            # Add noise offset for better training stability
            noise += self.config.noise_offset * torch.randn(
                (latents.shape[0], latents.shape[1], 1, 1), device=latents.device
            )
        
        bsz = latents.shape[0]
        # Sample a random timestep for each image
        timesteps = torch.randint(
            0, self.scheduler.config.num_train_timesteps, (bsz,), device=latents.device
        )
        timesteps = timesteps.long()
        
        # Add noise to the latents according to the noise magnitude at each timestep
        noisy_latents = self.scheduler.add_noise(latents, noise, timesteps)
        
        # Get the target for loss depending on the prediction type
        if self.scheduler.config.prediction_type == "epsilon":
            target = noise
        elif self.scheduler.config.prediction_type == "v_prediction":
            target = self.scheduler.get_velocity(latents, noise, timesteps)
        else:
            raise ValueError(f"Unknown prediction type {self.scheduler.config.prediction_type}")
        
        # Predict the noise residual and compute loss
        model_pred = self.unet(noisy_latents, timesteps, encoder_hidden_states).sample
        
        if self.config.snr_gamma is None:
            loss = F.mse_loss(model_pred.float(), target.float(), reduction="mean")
        else:
            # Compute loss-weights as per Section 3.4 of https://arxiv.org/abs/2303.09556.
            snr = self.compute_snr(timesteps)
            mse_loss_weights = (
                torch.stack([snr, self.config.snr_gamma * torch.ones_like(timesteps)], dim=1).min(dim=1)[0] / snr
            )
            loss = F.mse_loss(model_pred.float(), target.float(), reduction="none")
            loss = loss.mean(dim=list(range(1, len(loss.shape)))) * mse_loss_weights
            loss = loss.mean()
        
        return loss
    
    def compute_snr(self, timesteps):
        """Compute SNR for loss weighting"""
        alphas_cumprod = self.scheduler.alphas_cumprod
        sqrt_alphas_cumprod = alphas_cumprod**0.5
        sqrt_one_minus_alphas_cumprod = (1.0 - alphas_cumprod) ** 0.5
        
        sqrt_alphas_cumprod = sqrt_alphas_cumprod[timesteps].float()
        while len(sqrt_alphas_cumprod.shape) < len(timesteps.shape):
            sqrt_alphas_cumprod = sqrt_alphas_cumprod[..., None]
        sqrt_one_minus_alphas_cumprod = sqrt_one_minus_alphas_cumprod[timesteps].float()
        while len(sqrt_one_minus_alphas_cumprod.shape) < len(timesteps.shape):
            sqrt_one_minus_alphas_cumprod = sqrt_one_minus_alphas_cumprod[..., None]
        
        # Compute SNR.
        snr = (sqrt_alphas_cumprod / sqrt_one_minus_alphas_cumprod) ** 2
        return snr
    
    def validate(self, val_dataloader):
        """Run validation"""
        self.unet.eval()
        val_losses = []
        
        with torch.no_grad():
            for batch in val_dataloader:
                loss = self.compute_loss(batch)
                val_losses.append(loss.item())
        
        avg_val_loss = sum(val_losses) / len(val_losses)
        self.validation_losses.append(avg_val_loss)
        
        self.unet.train()
        return avg_val_loss
    
    def generate_validation_images(self, validation_prompts):
        """Generate validation images to monitor training progress"""
        from diffusers import StableDiffusionPipeline
        
        # Create pipeline with current state
        pipeline = StableDiffusionPipeline.from_pretrained(
            self.config.model_name,
            unet=self.accelerator.unwrap_model(self.unet),
            text_encoder=self.text_encoder,
            vae=self.vae,
            scheduler=self.scheduler,
            tokenizer=self.tokenizer,
            torch_dtype=self.weight_dtype,
            safety_checker=None,
            requires_safety_checker=False
        )
        pipeline = pipeline.to(self.accelerator.device)
        pipeline.set_progress_bar_config(disable=True)
        
        images = []
        for prompt in validation_prompts:
            with torch.autocast("cuda"):
                image = pipeline(
                    prompt,
                    num_inference_steps=25,
                    guidance_scale=7.5,
                    height=self.config.resolution,
                    width=self.config.resolution
                ).images[0]
            images.append(image)
        
        # Clean up
        del pipeline
        torch.cuda.empty_cache()
        
        return images
    
    def save_checkpoint(self, epoch, step, save_path):
        """Save model checkpoint"""
        save_path = Path(save_path)
        save_path.mkdir(parents=True, exist_ok=True)
        
        # Save LoRA weights
        self.accelerator.unwrap_model(self.unet).save_pretrained(save_path)
        
        # Save training state
        checkpoint = {
            'epoch': epoch,
            'global_step': step,
            'training_losses': self.training_losses,
            'validation_losses': self.validation_losses,
            'config': self.config.to_dict()
        }
        
        torch.save(checkpoint, save_path / "training_state.pt")
        logger.info(f"Checkpoint saved to {save_path}")
    
    def plot_training_progress(self):
        """Plot training and validation losses"""
        if len(self.training_losses) > 0:
            plt.figure(figsize=(12, 4))
            
            plt.subplot(1, 2, 1)
            plt.plot(self.training_losses)
            plt.title('Training Loss')
            plt.xlabel('Step')
            plt.ylabel('Loss')
            plt.grid(True)
            
            if len(self.validation_losses) > 0:
                plt.subplot(1, 2, 2)
                plt.plot(self.validation_losses)
                plt.title('Validation Loss')
                plt.xlabel('Validation Step')
                plt.ylabel('Loss')
                plt.grid(True)
            
            plt.tight_layout()
            plt.show()
    
    def train(self, train_dataloader, val_dataloader=None):
        """Main training loop"""
        
        # Setup optimizer and scheduler
        optimizer, lr_scheduler, max_train_steps = self.setup_optimizer_and_scheduler(train_dataloader)
        
        # Create output directories
        output_dir = Path(self.config.output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)
        
        # Validation prompts for monitoring
        validation_prompts = [
            f"a {self.config.model_name.split('/')[-1]} style calligraphy letter 'a', black ink on white background",
            f"a {self.config.model_name.split('/')[-1]} style calligraphy word 'hello', black ink on white background",
            f"a {self.config.model_name.split('/')[-1]} style calligraphy letter 'g', black ink on white background",
            f"a {self.config.model_name.split('/')[-1]} style calligraphy word 'world', black ink on white background"
        ]
        
        logger.info("***** Running training *****")
        logger.info(f"  Num examples = {len(train_dataloader.dataset)}")
        logger.info(f"  Num Epochs = {self.config.max_epochs}")
        logger.info(f"  Instantaneous batch size per device = {self.config.batch_size}")
        logger.info(f"  Total train batch size (w. parallel, distributed & accumulation) = {self.config.batch_size * self.accelerator.num_processes * self.config.gradient_accumulation_steps}")
        logger.info(f"  Gradient Accumulation steps = {self.config.gradient_accumulation_steps}")
        logger.info(f"  Total optimization steps = {max_train_steps}")
        
        self.global_step = 0
        first_epoch = 0
        print("🔥 Training loop entered")
        print(f"📦 Epoch count: {self.config.max_epochs}")
        # Training loop
        for epoch in range(first_epoch, self.config.max_epochs):
            self.unet.train()
            train_loss = 0.0
            
            for step, batch in enumerate(train_dataloader):
                with self.accelerator.accumulate(self.unet):
                    # Compute loss
                    loss = self.compute_loss(batch)
                    
                    # Gather the losses across all processes for logging
                    avg_loss = self.accelerator.gather(loss.repeat(self.config.batch_size)).mean()
                    train_loss += avg_loss.item() / self.config.gradient_accumulation_steps
                    
                    # Backpropagate
                    self.accelerator.backward(loss)
                    if self.accelerator.sync_gradients:
                        self.accelerator.clip_grad_norm_(self.unet.parameters(), self.config.max_grad_norm)
                    
                    optimizer.step()
                    lr_scheduler.step()
                    optimizer.zero_grad()
                
                # Checks if the accelerator has performed an optimization step behind the scenes
                if self.accelerator.sync_gradients:
                    self.global_step += 1
                    self.training_losses.append(train_loss)
                    
                    # Log training progress
                    if self.global_step % 50 == 0:
                        logger.info(f"Epoch {epoch}, Step {self.global_step}, Loss: {train_loss:.4f}, LR: {lr_scheduler.get_last_lr()[0]:.2e}")
                    
                    # Validation
                    if val_dataloader and self.global_step % self.config.validation_steps == 0:
                        val_loss = self.validate(val_dataloader)
                        logger.info(f"Validation Loss: {val_loss:.4f}")
                        
                        # Generate validation images
                        try:
                            val_images = self.generate_validation_images(validation_prompts[:2])  # Generate 2 images
                            # Display images
                            fig, axes = plt.subplots(1, 2, figsize=(10, 5))
                            for i, img in enumerate(val_images):
                                axes[i].imshow(img)
                                axes[i].set_title(f"Step {self.global_step}: {validation_prompts[i]}")
                                axes[i].axis('off')
                            plt.tight_layout()
                            plt.show()
                        except Exception as e:
                            logger.warning(f"Failed to generate validation images: {e}")
                    
                    # Save checkpoint
                    if self.global_step % self.config.save_steps == 0:
                        save_path = output_dir / f"checkpoint-{self.global_step}"
                        self.save_checkpoint(epoch, self.global_step, save_path)
                    
                    train_loss = 0.0
                
                if self.global_step >= max_train_steps:
                    break
            
            # End of epoch logging
            if val_dataloader:
                val_loss = self.validate(val_dataloader)
                logger.info(f"End of Epoch {epoch} - Validation Loss: {val_loss:.4f}")
        
        # Save final model
        final_save_path = output_dir / "final_model"
        self.save_checkpoint(self.config.max_epochs, self.global_step, final_save_path)
        
        # Plot training progress
        self.plot_training_progress()
        
        logger.info("Training completed!")
        return final_save_path

def create_trainer(config, models):
    """Factory function to create trainer"""
    return CalligraphyTrainer(config, models)

print("Training loop and optimization loaded successfully!")
print("Ready to start training your calligraphy LoRA model.")

Overwriting /kaggle/working/trainer.py


In [16]:
%%writefile /kaggle/working/main.py
#!/usr/bin/env python3

# main.py - Complete LoRA Training Script for Calligraphy Generation
# This script coordinates the entire training process for both simple and complex calligraphy styles

import os
import sys
import argparse
from pathlib import Path
import torch
import gc
from datetime import datetime
import json
import shutil

# Import our custom modules (assuming they're in the same directory or properly installed)
# If running in Kaggle, make sure all the other .py files are uploaded
try:
    from setup import *  # This imports all the basic setup, logging, etc.
    from data_preparation import ImageProcessor, DatasetBuilder
    from dataset import CalligraphyDataset, create_data_loaders, preview_dataset
    from model_setup import LoRAModelSetup, TrainingConfig, calculate_memory_usage
    from trainer import CalligraphyTrainer, create_trainer
except ImportError as e:
    print(f"Import error: {e}")
    print("Make sure all the component files (setup.py, data_preparation.py, etc.) are in the same directory")
    sys.exit(1)

class CalligraphyTrainingPipeline:
    """Main pipeline for calligraphy LoRA training"""
    
    def __init__(self, 
                 character_dir: str,
                 word_dir: str,
                 style_name: str = "calligraphy",
                 output_base_dir: str = "/kaggle/working",
                 complex_style: bool = False):
        
        self.character_dir = Path(character_dir)
        self.word_dir = Path(word_dir)
        self.style_name = style_name
        self.output_base_dir = Path(output_base_dir)
        self.complex_style = complex_style
        
        # Create timestamped output directory
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        self.experiment_dir = self.output_base_dir / f"{style_name}_lora_{timestamp}"
        self.data_dir = self.experiment_dir / "processed_data"
        self.model_dir = self.experiment_dir / "model"
        self.logs_dir = self.experiment_dir / "logs"
        
        # Create directories
        for dir_path in [self.experiment_dir, self.data_dir, self.model_dir, self.logs_dir]:
            dir_path.mkdir(parents=True, exist_ok=True)
        
        logger.info(f"Initialized training pipeline for {style_name}")
        logger.info(f"Output directory: {self.experiment_dir}")
        
    def validate_input_data(self):
        """Validate input directories and data"""
        print("Code is reaching here !!")
        print("Here is character dir : ",self.character_dir)
        print("Here is character dir with exists : ",self.character_dir.exists()) 
        if not self.character_dir.exists():
            raise FileNotFoundError(f"Character directory not found: {self.character_dir}")
        print("Here is word dir : ",self.word_dir)
        print("Here is word dir with exists : ",self.word_dir.exists()) 
        if not self.word_dir.exists():
            raise FileNotFoundError(f"Word directory not found: {self.word_dir}")
        print("Code is here")
        # Count available images
        char_images = list(self.character_dir.glob("*.png")) + list(self.character_dir.glob("*.jpg"))
        word_images = list(self.word_dir.glob("*.png")) + list(self.word_dir.glob("*.jpg"))
        print("Code has two things here")
        logger.info(f"Found {len(char_images)} character images")
        logger.info(f"Found {len(word_images)} word images")
        
        if len(char_images) == 0 and len(word_images) == 0:
            raise ValueError("No training images found in the specified directories")
        
        return len(char_images), len(word_images)
    
    def setup_config(self, char_count: int, word_count: int):
        """Setup training configuration based on data and hardware"""
        config = TrainingConfig()
        
        # Adjust configuration based on complexity and data size
        total_samples = char_count + word_count
        
        if self.complex_style:
            # More aggressive training for complex styles
            config.lora_rank = 8  # Higher rank for complex patterns
            config.lora_alpha = 64
            config.learning_rate = 5e-5  # Lower learning rate for stability
            config.max_epochs = 100  # More epochs for complex styles
            config.noise_offset = 0.15  # Higher noise offset
            config.style_name = f"complex_{self.style_name}"
        else:
            # Standard configuration for simple styles
            config.lora_rank = 4
            config.lora_alpha = 32
            config.learning_rate = 1e-4
            config.max_epochs = 50
            config.noise_offset = 0.1
            config.style_name = self.style_name
        
        # Adjust batch size and epochs based on data size
        if total_samples < 50:
            config.batch_size = 1
            config.gradient_accumulation_steps = 8
            config.max_epochs = max(config.max_epochs, 80)  # More epochs for small datasets
        elif total_samples > 200:
            config.batch_size = 2 if torch.cuda.is_available() else 1
            config.gradient_accumulation_steps = 2
        
        # Update paths
        config.output_dir = str(self.model_dir)
        config.logging_dir = str(self.logs_dir)
        
        logger.info(f"Configuration setup complete:")
        logger.info(f"  LoRA rank: {config.lora_rank}")
        logger.info(f"  Learning rate: {config.learning_rate}")
        logger.info(f"  Max epochs: {config.max_epochs}")
        logger.info(f"  Batch size: {config.batch_size}")
        
        return config
    
    def prepare_data(self):
        """Prepare training data"""
        logger.info("Starting data preparation...")
        
        # Initialize dataset builder
        dataset_builder = DatasetBuilder(style_name=self.style_name)
        
        # Create training data
        training_info = dataset_builder.create_training_data(
            character_dir=str(self.character_dir),
            word_dir=str(self.word_dir),
            output_dir=str(self.data_dir)
        )
        
        logger.info(f"Data preparation complete: {training_info['total_samples']} samples created")
        
        return training_info
    
    def setup_models(self, config):
        """Setup LoRA models"""
        logger.info("Setting up models...")
        
        # Initialize model setup
        model_setup = LoRAModelSetup(
            model_name=config.model_name,
            revision=config.revision,
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
        )
        
        # Load base models
        models = model_setup.load_models()
        
        # Setup LoRA configuration
        lora_config = model_setup.setup_lora_config(
            rank=config.lora_rank,
            alpha=config.lora_alpha,
            dropout=config.lora_dropout
        )
        
        # Apply LoRA to UNet
        models["unet"] = model_setup.apply_lora_to_unet(lora_config)
        
        logger.info("Models setup complete")
        
        return models
        
    def create_data_loaders(self, training_info, config, models):
        """Create training and validation data loaders"""
        logger.info("Creating data loaders...")
        
        train_loader, val_loader = create_data_loaders(
            images_dir=training_info["images_dir"],
            metadata_path=training_info["metadata_path"],
            tokenizer=models["tokenizer"],
            batch_size=config.batch_size,
            validation_split=config.validation_split,
            resolution=config.resolution
        )
        
        logger.info(f"Data loaders created: {len(train_loader)} train batches, {len(val_loader)} val batches")
        
        return train_loader, val_loader
    
    def train_model(self, config, models, train_loader, val_loader):
        """Train the LoRA model"""
        
        print("✅ BEGIN TRAINING DIAGNOSTICS")
        print(f"Epochs to train: {config.max_epochs}")
        print(f"Train batches: {len(train_loader)}")
        print(f"Validation batches: {len(val_loader)}")
        print(f"Batch size: {config.batch_size}")
        print("✅ END DIAGNOSTICS")
        
        logger.info("Starting model training...")
        
        # Create trainer
        trainer = create_trainer(config, models)
        
        # Start training
        final_model_path = trainer.train(train_loader, val_loader)
        
        logger.info(f"Training complete! Final model saved to: {final_model_path}")
        
        return final_model_path
    
    def create_inference_pipeline(self, final_model_path, config):
        """Create inference pipeline for testing"""
        logger.info("Creating inference pipeline...")
        
        try:
            from diffusers import StableDiffusionPipeline
            
            # Load the trained LoRA model
            pipeline = StableDiffusionPipeline.from_pretrained(
                config.model_name,
                torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
                safety_checker=None,
                requires_safety_checker=False
            )
            
            # Load LoRA weights
            pipeline.unet.load_adapter(final_model_path)
            
            if torch.cuda.is_available():
                pipeline = pipeline.to("cuda")
            
            # Save pipeline for later use
            pipeline_save_path = self.experiment_dir / "inference_pipeline"
            pipeline.save_pretrained(pipeline_save_path)
            
            logger.info(f"Inference pipeline saved to: {pipeline_save_path}")
            
            return pipeline, pipeline_save_path
            
        except Exception as e:
            logger.warning(f"Failed to create inference pipeline: {e}")
            return None, None
    
    def test_generation(self, pipeline, config):
        """Test the trained model with sample prompts"""
        if pipeline is None:
            logger.warning("No pipeline available for testing")
            return
        
        logger.info("Testing model with sample prompts...")
        
        # Test prompts based on style complexity
        if self.complex_style:
            test_prompts = [
                f"a {config.style_name} gothic calligraphy letter 'd', black ink on white background, ornate medieval style",
                f"a {config.style_name} gothic calligraphy word 'dog', black ink on white background, decorative lettering",
                f"a {config.style_name} gothic calligraphy letter 'A', black ink on white background, elaborate flourishes",
                f"a {config.style_name} gothic calligraphy word 'fate', black ink on white background, medieval manuscript style"
            ]
        else:
            test_prompts = [
                f"a {config.style_name} calligraphy letter 'a', black ink on white background, elegant handwriting",
                f"a {config.style_name} calligraphy word 'hello', black ink on white background, flowing script",
                f"a {config.style_name} calligraphy letter 'g', black ink on white background, cursive style",
                f"a {config.style_name} calligraphy word 'world', black ink on white background, beautiful lettering"
            ]
        
        test_results_dir = self.experiment_dir / "test_results"
        test_results_dir.mkdir(exist_ok=True)
        
        try:
            for i, prompt in enumerate(test_prompts):
                logger.info(f"Generating: {prompt}")
                
                with torch.autocast("cuda" if torch.cuda.is_available() else "cpu"):
                    image = pipeline(
                        prompt,
                        num_inference_steps=30,
                        guidance_scale=7.5,
                        height=config.resolution,
                        width=config.resolution,
                        generator=torch.Generator().manual_seed(42)  # For reproducible results
                    ).images[0]
                
                # Save result
                image.save(test_results_dir / f"test_{i+1:02d}.png")
                
                # Also save prompt for reference
                with open(test_results_dir / f"test_{i+1:02d}_prompt.txt", "w") as f:
                    f.write(prompt)
            
            logger.info(f"Test results saved to: {test_results_dir}")
            
        except Exception as e:
            logger.error(f"Error during testing: {e}")
    
    def save_experiment_info(self, config, training_info, final_model_path):
        """Save experiment information and metadata"""
        experiment_info = {
            "experiment_name": f"{self.style_name}_lora_training",
            "timestamp": datetime.now().isoformat(),
            "style_name": self.style_name,
            "complex_style": self.complex_style,
            "input_directories": {
                "character_dir": str(self.character_dir),
                "word_dir": str(self.word_dir)
            },
            "output_directories": {
                "experiment_dir": str(self.experiment_dir),
                "model_dir": str(self.model_dir),
                "data_dir": str(self.data_dir)
            },
            "training_data": training_info,
            "final_model_path": str(final_model_path),
            "config": config.to_dict(),
            "hardware_info": {
                "cuda_available": torch.cuda.is_available(),
                "device_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
            }
        }
        
        # Save experiment info
        info_path = self.experiment_dir / "experiment_info.json"
        with open(info_path, "w") as f:
            json.dump(experiment_info, f, indent=2)
        
        logger.info(f"Experiment info saved to: {info_path}")
        
        return experiment_info
    
    def run_complete_training(self):
        """Run the complete training pipeline"""
        print("Complete Training Called!!")
        logger.info("=" * 80)
        logger.info(f"STARTING CALLIGRAPHY LORA TRAINING PIPELINE")
        logger.info(f"Style: {self.style_name}")
        logger.info(f"Complex Style: {self.complex_style}")
        logger.info("=" * 80)
        
        try:
            # Step 1: Validate input data
            print("Validate_Input_Data")
            char_count, word_count = self.validate_input_data()
            
            # Step 2: Setup configuration
            print("Validate_Setup")
            config = self.setup_config(char_count, word_count)
            
            # Step 3: Check memory
            logger.info("Initial memory status:")
            calculate_memory_usage()
            
            # Step 4: Prepare data
            training_info = self.prepare_data()
            
            # Step 5: Setup models
            models = self.setup_models(config)
            
            # Step 6: Create data loaders
            train_loader, val_loader = self.create_data_loaders(training_info, config, models)
            
            # Step 7: Preview dataset (optional)
            logger.info("Previewing dataset...")
            try:
                dataset = CalligraphyDataset(
                    images_dir=training_info["images_dir"],
                    metadata_path=training_info["metadata_path"],
                    tokenizer=models["tokenizer"],
                    resolution=config.resolution
                )
                preview_dataset(dataset, num_samples=3)
            except Exception as e:
                logger.warning(f"Failed to preview dataset: {e}")
            
            # Step 8: Train model
            final_model_path = self.train_model(config, models, train_loader, val_loader)
            
            # Step 9: Create inference pipeline
            pipeline, pipeline_path = self.create_inference_pipeline(final_model_path, config)
            
            # Step 10: Test generation
            self.test_generation(pipeline, config)
            
            # Step 11: Save experiment info
            experiment_info = self.save_experiment_info(config, training_info, final_model_path)
            
            # Clean up memory
            del models, train_loader, val_loader
            if pipeline:
                del pipeline
            gc.collect()
            torch.cuda.empty_cache() if torch.cuda.is_available() else None
            
            logger.info("=" * 80)
            logger.info("TRAINING PIPELINE COMPLETED SUCCESSFULLY!")
            logger.info(f"Results saved to: {self.experiment_dir}")
            logger.info(f"Final model: {final_model_path}")
            logger.info("=" * 80)
            
            return {
                "success": True,
                "experiment_dir": str(self.experiment_dir),
                "final_model_path": str(final_model_path),
                "pipeline_path": str(pipeline_path) if pipeline_path else None,
                "experiment_info": experiment_info
            }
            
        except Exception as e:
            logger.error(f"Training pipeline failed: {e}")
            logger.error(f"Error type: {type(e).__name__}")
            import traceback
            logger.error(f"Traceback:\n{traceback.format_exc()}")
            
            return {
                "success": False,
                "error": str(e),
                "experiment_dir": str(self.experiment_dir)
            }

def main():
    """Main function with command line interface"""
    
    # Create and run training pipeline
    pipeline = CalligraphyTrainingPipeline(
        character_dir='/kaggle/input/character-simple-images',
        word_dir='/kaggle/input/word-simple-images',
        style_name="calligraphy",
        output_base_dir='/kaggle/working/output-data',
        complex_style=False
    )
    
    result = pipeline.run_complete_training()
    if result["success"]:
        print("\n" + "="*50)
        print("🎉 TRAINING COMPLETED SUCCESSFULLY! 🎉")
        print("="*50)
        print(f"📁 Results: {result['experiment_dir']}")
        print(f"🤖 Model: {result['final_model_path']}")
        if result['pipeline_path']:
            print(f"🔧 Pipeline: {result['pipeline_path']}")
        print("="*50)
    else:
        print("\n" + "="*50)
        print("❌ TRAINING FAILED")
        print("="*50)
        print(f"Error: {result['error']}")
        print(f"Check logs in: {result['experiment_dir']}")
        print("="*50)
        sys.exit(1)

# Convenience function for Jupyter notebook usage
def train_calligraphy_model(character_dir: str, 
                           word_dir: str, 
                           style_name: str = "calligraphy",
                           complex_style: bool = False,
                           output_dir: str = "/kaggle/working"):
    """
    Convenience function for training in Jupyter notebooks
    
    Args:
        character_dir: Path to directory with character images
        word_dir: Path to directory with word images  
        style_name: Name of the calligraphy style
        complex_style: Whether to use complex style settings
        output_dir: Base output directory
    
    Returns:
        Training results dictionary
    """
    pipeline = CalligraphyTrainingPipeline(
        character_dir=character_dir,
        word_dir=word_dir,
        style_name=style_name,
        output_base_dir=output_dir,
        complex_style=complex_style
    )
    
    return pipeline.run_complete_training()

if __name__ == "__main__":
    main()


"""  
result = train_calligraphy_model(
    character_dir="/kaggle/input/character-simple-images",
    word_dir="/kaggle/input/word-simple-images", 
    style_name="elegant_script",
    complex_style=False
)

# For simple calligraphy (like your first image):

# Example usage for Kaggle/Jupyter:

# For complex/gothic calligraphy (like your second image):
result = train_calligraphy_model(
    character_dir="/kaggle/input/your-data/character_images",
    word_dir="/kaggle/input/your-data/word_images",
    style_name="gothic_calligraphy", 
    complex_style=True
)
"""

Overwriting /kaggle/working/main.py


In [17]:
from main import train_calligraphy_model

result = train_calligraphy_model(
    character_dir="/kaggle/input/calligraphy-simple-images",   # replace with your actual path
    word_dir="kaggle/input/word-simple-images",             # replace with your actual path
    style_name="simple",
    complex_style=False,                         # set to True for complex calligraphy
    output_dir="/kaggle/working/output-data"            # optional
)

Complete Training Called!!
Validate_Input_Data
Code is reaching here !!


In [ ]:
"Hello Everyone, this is a good morning !!, in gothic calligraphy style, black ink on white paper"

In [18]:
#!/usr/bin/env python3
"""
Debug script for calligraphy training - identifies common failure points
"""

import os
import sys
import traceback
from pathlib import Path
import torch
import logging

# Setup detailed logging
logging.basicConfig(
    level=logging.DEBUG,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[
        logging.StreamHandler(sys.stdout),
        logging.FileHandler('/kaggle/working/debug.log')
    ]
)
logger = logging.getLogger(__name__)

def check_environment():
    """Check if environment is properly set up"""
    print("🔍 CHECKING ENVIRONMENT...")
    
    # Check Python version
    print(f"Python version: {sys.version}")
    
    # Check CUDA availability
    cuda_available = torch.cuda.is_available()
    print(f"CUDA available: {cuda_available}")
    if cuda_available:
        print(f"CUDA device: {torch.cuda.get_device_name(0)}")
        print(f"CUDA memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    
    # Check required packages
    required_packages = [
        'torch', 'torchvision', 'transformers', 'diffusers', 
        'accelerate', 'peft', 'opencv-python', 'pillow'
    ]
    
    missing_packages = []
    for package in required_packages:
        try:
            __import__(package.replace('-', '_'))
            print(f"✅ {package}")
        except ImportError:
            print(f"❌ {package} - MISSING!")
            missing_packages.append(package)
    
    if missing_packages:
        print(f"\n🚨 Missing packages: {missing_packages}")
        return False
    
    return True

def check_input_directories(character_dir, word_dir):
    """Check if input directories exist and contain data"""
    print("\n🔍 CHECKING INPUT DIRECTORIES...")
    
    char_path = Path(character_dir)
    word_path = Path(word_dir)
    
    # Check character directory
    if not char_path.exists():
        print(f"❌ Character directory does not exist: {character_dir}")
        return False
    
    char_images = list(char_path.glob("*.png")) + list(char_path.glob("*.jpg")) + list(char_path.glob("*.jpeg"))
    print(f"📁 Character directory: {character_dir}")
    print(f"   Found {len(char_images)} images")
    if len(char_images) > 0:
        print(f"   Sample files: {[f.name for f in char_images[:3]]}")
    
    # Check word directory  
    if not word_path.exists():
        print(f"❌ Word directory does not exist: {word_dir}")
        return False
    
    word_images = list(word_path.glob("*.png")) + list(word_path.glob("*.jpg")) + list(word_path.glob("*.jpeg"))
    print(f"📁 Word directory: {word_dir}")
    print(f"   Found {len(word_images)} images")
    if len(word_images) > 0:
        print(f"   Sample files: {[f.name for f in word_images[:3]]}")
    
    total_images = len(char_images) + len(word_images)
    if total_images == 0:
        print("❌ No images found in either directory!")
        return False
    
    print(f"✅ Total images found: {total_images}")
    return True

def check_output_directory(output_dir):
    """Check if output directory is writable"""
    print("\n🔍 CHECKING OUTPUT DIRECTORY...")
    
    output_path = Path(output_dir)
    
    try:
        output_path.mkdir(parents=True, exist_ok=True)
        
        # Test write permissions
        test_file = output_path / "test_write.txt"
        test_file.write_text("test")
        test_file.unlink()
        
        print(f"✅ Output directory is writable: {output_dir}")
        return True
        
    except Exception as e:
        print(f"❌ Cannot write to output directory: {e}")
        return False

def check_imports():
    """Check if all custom modules can be imported"""
    print("\n🔍 CHECKING CUSTOM IMPORTS...")
    
    modules_to_check = [
        'setup',
        'data_preparation', 
        'dataset',
        'model_setup',
        'trainer'
    ]
    
    failed_imports = []
    
    for module in modules_to_check:
        try:
            exec(f"import {module}")
            print(f"✅ {module}")
        except ImportError as e:
            print(f"❌ {module} - {e}")
            failed_imports.append(module)
        except Exception as e:
            print(f"⚠️ {module} - Error: {e}")
            failed_imports.append(module)
    
    if failed_imports:
        print(f"\n🚨 Failed to import: {failed_imports}")
        return False
    
    return True

def test_data_preparation(character_dir, word_dir, output_dir):
    """Test data preparation step"""
    print("\n🔍 TESTING DATA PREPARATION...")
    
    try:
        from data_preparation import DatasetBuilder
        
        # Create a small test
        test_output = Path(output_dir) / "test_data_prep"
        test_output.mkdir(parents=True, exist_ok=True)
        
        dataset_builder = DatasetBuilder(style_name="test_style")
        
        # Try to process just one image from each directory
        char_path = Path(character_dir)
        word_path = Path(word_dir)
        
        char_images = list(char_path.glob("*.png"))[:1]  # Just first image
        word_images = list(word_path.glob("*.png"))[:1]  # Just first image
        
        if char_images:
            print(f"   Testing character extraction from: {char_images[0].name}")
            char = dataset_builder.extract_character_from_filename(char_images[0].name)
            print(f"   Extracted character: '{char}'")
        
        if word_images:
            print(f"   Testing word extraction from: {word_images[0].name}")
            word = dataset_builder.extract_word_from_filename(word_images[0].name)
            print(f"   Extracted word: '{word}'")
        
        print("✅ Data preparation test passed")
        return True
        
    except Exception as e:
        print(f"❌ Data preparation test failed: {e}")
        traceback.print_exc()
        return False

def test_model_setup():
    """Test model setup"""
    print("\n🔍 TESTING MODEL SETUP...")
    
    try:
        from model_setup import TrainingConfig, LoRAModelSetup
        
        # Test config creation
        config = TrainingConfig()
        print(f"   Config created: {config.model_name}")
        
        # Test model loading (this might take time)
        print("   Testing model loading (this may take a moment)...")
        model_setup = LoRAModelSetup(
            model_name=config.model_name,
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
        )
        
        # Just test tokenizer loading
        from transformers import CLIPTokenizer
        tokenizer = CLIPTokenizer.from_pretrained(
            config.model_name, 
            subfolder="tokenizer"
        )
        print(f"   Tokenizer loaded: {tokenizer.__class__.__name__}")
        
        print("✅ Model setup test passed")
        return True
        
    except Exception as e:
        print(f"❌ Model setup test failed: {e}")
        traceback.print_exc()
        return False

def debug_training_call(character_dir, word_dir, style_name, complex_style, output_dir):
    """Debug the actual training call step by step"""
    print("\n🔍 DEBUGGING TRAINING CALL...")
    
    try:
        # Import main function
        print("   Importing train_calligraphy_model...")
        from main import train_calligraphy_model
        print("   ✅ Import successful")
        
        # Create the pipeline object manually to test each step
        print("   Creating CalligraphyTrainingPipeline...")
        from main import CalligraphyTrainingPipeline
        
        pipeline = CalligraphyTrainingPipeline(
            character_dir=character_dir,
            word_dir=word_dir,
            style_name=style_name,
            output_base_dir=output_dir,
            complex_style=complex_style
        )
        print("   ✅ Pipeline created")
        
        # Test validation step
        print("   Testing input validation...")
        char_count, word_count = pipeline.validate_input_data()
        print(f"   ✅ Validation passed: {char_count} chars, {word_count} words")
        
        # Test config setup
        print("   Testing config setup...")
        config = pipeline.setup_config(char_count, word_count)
        print(f"   ✅ Config created: {config.style_name}")
        
        # Test data preparation
        print("   Testing data preparation...")
        training_info = pipeline.prepare_data()
        print(f"   ✅ Data prepared: {training_info['total_samples']} samples")
        
        print("✅ Training call debug completed successfully")
        return True
        
    except Exception as e:
        print(f"❌ Training call debug failed: {e}")
        traceback.print_exc()
        return False

def run_complete_debug(character_dir, word_dir, style_name="simple", complex_style=False, output_dir="/kaggle/working/output-data"):
    """Run complete debugging process"""
    print("🚀 STARTING COMPLETE DEBUG PROCESS")
    print("=" * 60)
    
    # Step 1: Environment check
    if not check_environment():
        print("❌ Environment check failed!")
        return False
    
    # Step 2: Input directory check
    if not check_input_directories(character_dir, word_dir):
        print("❌ Input directory check failed!")
        return False
    
    # Step 3: Output directory check
    if not check_output_directory(output_dir):
        print("❌ Output directory check failed!")
        return False
    
    # Step 4: Import check
    if not check_imports():
        print("❌ Import check failed!")
        return False
    
    # Step 5: Data preparation test
    if not test_data_preparation(character_dir, word_dir, output_dir):
        print("❌ Data preparation test failed!")
        return False
    
    # Step 6: Model setup test
    if not test_model_setup():
        print("❌ Model setup test failed!")
        return False
    
    # Step 7: Training call debug
    if not debug_training_call(character_dir, word_dir, style_name, complex_style, output_dir):
        print("❌ Training call debug failed!")
        return False
    
    print("\n" + "=" * 60)
    print("🎉 ALL DEBUG CHECKS PASSED!")
    print("Your environment should be ready for training.")
    print("=" * 60)
    
    return True

# Example usage for your specific case
if __name__ == "__main__":
    # Fix the path issue in your original code (missing leading slash)
    result = run_complete_debug(
        character_dir="/kaggle/input/calligraphy-simple-images",
        word_dir="/kaggle/input/word-simple-images",  # Fixed: was missing leading slash
        style_name="simple",
        complex_style=False,
        output_dir="/kaggle/working/output-data"
    )
    
    if result:
        print("\n🚀 Attempting actual training call...")
        try:
            from main import train_calligraphy_model
            
            result = train_calligraphy_model(
                character_dir="/kaggle/input/calligraphy-simple-images",
                word_dir="/kaggle/input/word-simple-images",  # Fixed path
                style_name="simple",
                complex_style=False,
                output_dir="/kaggle/working/output-data"
            )
            
            print(f"Training result: {result}")
            
        except Exception as e:
            print(f"❌ Training failed with error: {e}")
            traceback.print_exc()
    else:
        print("❌ Debug checks failed - fix issues before training")

🚀 STARTING COMPLETE DEBUG PROCESS
🔍 CHECKING ENVIRONMENT...
Python version: 3.11.11 (main, Dec  4 2024, 08:55:07) [GCC 11.4.0]
CUDA available: True
CUDA device: Tesla T4
CUDA memory: 15.8 GB
✅ torch
✅ torchvision
✅ transformers
✅ diffusers
✅ accelerate
✅ peft
❌ opencv-python - MISSING!
❌ pillow - MISSING!

🚨 Missing packages: ['opencv-python', 'pillow']
❌ Environment check failed!
❌ Debug checks failed - fix issues before training
